# 03 — Workout Recommender Training**Goal:** build a content-based exercise recommender that, given a user's fitness level (and optionally their goal + activity preference), returns a ranked list of exercises with rationales.**Algorithm:** TF-IDF over (exercise, muscle_group, difficulty, level) tokens, then a cosine-similarity look-up against the candidate catalog. We also persist a per-fitness-level catalog so the API can serve top-K instantly without re-running the model at request time.**Output artifact:** `ai_models/ml_models/recommender.pkl` — a dict `{0: [...beginner exercises], 1: [...intermediate], 2: [...advanced]}`. This is what `MLService.recommend_exercises` reads.**Dataset:** `datasets/workout_history.csv` — 8,000 rows of `(user_id, exercise, muscle_group, difficulty, duration_min, rating, fitness_level)`.

In [ ]:
import sys, pathlib, json, timeROOT = pathlib.Path('..').resolve()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport joblibfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similaritysns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (10, 5)print('imports OK')

## 1. Load + explore

In [ ]:
df = pd.read_csv(ROOT / 'datasets' / 'workout_history.csv')print(f'Loaded {len(df)} workout-history rows')df.head()

In [ ]:
# Most popular exercises per fitness leveltop_per_level = (df.groupby(['fitness_level','exercise'])                   .agg(n=('rating','size'), avg_rating=('rating','mean'))                   .reset_index())fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))for ax, lvl, lbl in zip(axes, [0,1,2], ['Beginner','Intermediate','Advanced']):    top = top_per_level[top_per_level['fitness_level']==lvl].nlargest(8, 'n')    sns.barplot(data=top, y='exercise', x='n', ax=ax, palette='viridis')    ax.set_title(f'{lbl} · top exercises'); ax.set_xlabel('# logged')plt.tight_layout(); plt.show()

## 2. Build the candidate catalogFor each exercise we compute aggregate stats across users — popularity, average rating, average duration. These are what the recommender ranks on, after similarity filtering.

In [ ]:
catalog = (df.groupby(['exercise','muscle_group','difficulty','fitness_level'])              .agg(n_logs=('rating','size'),                   avg_rating=('rating','mean'),                   avg_duration=('duration_min','mean'))              .reset_index())catalog['avg_rating']   = catalog['avg_rating'].round(2)catalog['avg_duration'] = catalog['avg_duration'].round(1)print(f'Catalog: {len(catalog)} (exercise × muscle_group × difficulty × level) rows')catalog.head()

## 3. TF-IDF feature space

In [ ]:
# Build a 'document' per row that captures the exercise's identity + tags.def make_doc(row):    parts = [        row['exercise'].lower(),        row['muscle_group'],        f"diff{int(row['difficulty'])}",        f"level{int(row['fitness_level'])}",    ]    return ' '.join(parts)catalog['doc'] = catalog.apply(make_doc, axis=1)vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)M = vec.fit_transform(catalog['doc'])print(f'TF-IDF matrix: {M.shape}  (rows × tokens)')# Quick cosine-similarity self-check: each row's nearest neighbour should be itselfsim = cosine_similarity(M[:1], M)[0]top_idx = np.argsort(-sim)[:5]print('Top 5 nearest to row 0:'); print(catalog.iloc[top_idx][['exercise','muscle_group','fitness_level']])

## 4. Build the per-level recommendation catalogueThe API doesn't need the TF-IDF model at runtime — it just needs an ordered list per level. We score each exercise by `popularity × rating` and pre-compute the ranking.

In [ ]:
def score(row):    # Geometric mean of normalised popularity and rating, slightly favouring rating.    pop_n = (row['n_logs'] / catalog['n_logs'].max())    rat_n = (row['avg_rating'] / 5.0)    return float((pop_n ** 0.4) * (rat_n ** 0.6))catalog['score'] = catalog.apply(score, axis=1)per_level = {}for lvl in [0, 1, 2]:    sub = (catalog[catalog['fitness_level']==lvl]              .sort_values('score', ascending=False)              .head(20)              .reset_index(drop=True))    per_level[lvl] = sub[[        'exercise','muscle_group','difficulty','avg_rating','avg_duration','score','n_logs'    ]].to_dict('records')for lvl, items in per_level.items():    print(f'\nLevel {lvl} top 5:')    for it in items[:5]:        print(f"  · {it['exercise']:25s} ({it['muscle_group']:8s}) ★{it['avg_rating']}  score={it['score']:.3f}")

## 5. Diversity checkThe recommender shouldn't return five chest exercises in a row. We measure muscle-group entropy in the top-5 of each level.

In [ ]:
from collections import Counterdef diversity(items, k=5):    groups = [it['muscle_group'] for it in items[:k]]    c = Counter(groups)    p = np.array(list(c.values())) / sum(c.values())    return float(-(p * np.log(p)).sum())  # natsfor lvl, items in per_level.items():    print(f'Level {lvl} top-5 muscle-group entropy: {diversity(items):.3f}  '          f"(groups: {[it['muscle_group'] for it in items[:5]]})")

## 6. Save artefacts

In [ ]:
out_dir = ROOT / 'ai_models' / 'ml_models'out_dir.mkdir(parents=True, exist_ok=True)# The dict shape MLService expects:joblib.dump(per_level, out_dir / 'recommender.pkl')# Optional: save the TF-IDF model too in case we want online similarity laterjoblib.dump({'vectorizer': vec, 'catalog': catalog}, out_dir / 'recommender_tfidf.pkl')report = {    'model': 'recommender',    'algorithm': 'TF-IDF + popularity-weighted ranking, per fitness level',    'n_catalog_rows': int(len(catalog)),    'top_k_per_level': 20,    'levels': sorted(per_level.keys()),    'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),}(out_dir / 'reports').mkdir(exist_ok=True)(out_dir / 'reports' / f'recommender_{time.strftime("%Y%m%d_%H%M%S")}.json').write_text(json.dumps(report, indent=2))print('saved recommender.pkl + recommender_tfidf.pkl + report')

## 7. Smoke-test the API path

In [ ]:
import backend.services.ml_service as ml_modml_mod.get_ml_service.cache_clear()from backend.services.ml_service import get_ml_servicesvc = get_ml_service()for lvl, name in [(0,'Beginner'),(1,'Intermediate'),(2,'Advanced')]:    items = svc.recommend_exercises(level_id=lvl, top_k=5,                                    profile={'goal':'lose'})    print(f'\n— {name} (goal=lose) —')    for i in items:        print(f"  · {i.get('exercise'):25s} — {i.get('rationale','')}")

---The `/recommend` endpoint will now use this exact catalogue. Re-run the cells whenever you've added a meaningful amount of new workout-history data.